# MIT805 Part 2 - MapReduce & Visualization with PySpark

This notebook is a lightweight PySpark scaffold for the full NYC TLC HVFHV dataset. It is intentionally focused on distributed processing evidence and two example queries using the real data, rather than providing a full EDA.

## Assignment requirement

Students must also provide evidence of Spark execution through an execution plan, Spark UI, DAG/stage information, or equivalent evidence. Students must distinguish Spark's DAG-based execution model from the traditional Map → Shuffle → Reduce model of Hadoop MapReduce.

This notebook includes a small SQL section and a Spark execution-plan example to satisfy that requirement.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = (
    SparkSession.builder
    .appName('MIT805-Part2-HVFHV')
    .master('local[*]')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.adaptive.advisoryPartitionSizeInBytes', '64m')
    .config('spark.sql.shuffle.partitions', '200')
    .getOrCreate()
)

spark


In [ ]:
base_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data'

def build_urls(start_year=2019, start_month=2, end_year=2026, end_month=6):
    urls = []
    for year in range(start_year, end_year + 1):
        month_start = 1
        month_end = 12
        if year == start_year:
            month_start = start_month
        if year == end_year:
            month_end = end_month
        for month in range(month_start, month_end + 1):
            urls.append(f'{base_url}/fhvhv_tripdata_{year}-{month:02d}.parquet')
    return urls

urls = build_urls()
print(f'Loading {len(urls)} parquet files...')

hvfhv = (
    spark.read
    .option('mergeSchema', 'true')
    .parquet(*urls)
)

print(f'Rows: {hvfhv.count():,}')
print(f'Columns: {len(hvfhv.columns)}')
hvfhv.printSchema()


## Processing objective

The project is using the full HVFHV dataset to examine how trip activity varies over time and by operator. The central question is: what are the operational patterns in dispatch demand, trip volume, and trip value across the full dataset?

This is a distributed data-processing problem because the dataset spans many monthly parquet files and exceeds the limits of a single-machine workflow.

In [ ]:
# Example query 1: monthly trip count using a MapReduce-style transformation
monthly_volume = (
    hvfhv
    .withColumn('pickup_date', F.to_date('pickup_datetime'))
    .groupBy(F.trunc('pickup_date', 'MM').alias('pickup_month'))
    .agg(F.count('*').alias('trip_count'))
    .orderBy('pickup_month')
)

monthly_volume.show(12, truncate=False)

In [ ]:
# Example query 2: operator summary including average trip distance and fare
operator_summary = (
    hvfhv
    .filter(F.col('pickup_datetime').isNotNull())
    .groupBy('hvfhs_license_num')
    .agg(
        F.count('*').alias('trip_count'),
        F.avg('trip_miles').alias('avg_trip_miles'),
        F.avg('base_passenger_fare').alias('avg_base_fare')
    )
    .orderBy(F.col('trip_count').desc())
)

operator_summary.show(10, truncate=False)

In [ ]:
# Small Spark SQL section using the same data
hvfhv.createOrReplaceTempView('hvfhv')

spark_sql_result = spark.sql("""
    SELECT
        DATE_TRUNC('month', CAST(pickup_datetime AS TIMESTAMP)) AS pickup_month,
        COUNT(*) AS trip_count,
        ROUND(AVG(trip_miles), 2) AS avg_trip_miles,
        ROUND(AVG(base_passenger_fare), 2) AS avg_base_fare
    FROM hvfhv
    WHERE pickup_datetime IS NOT NULL
    GROUP BY DATE_TRUNC('month', CAST(pickup_datetime AS TIMESTAMP))
    ORDER BY pickup_month
""")

spark_sql_result.show(12, truncate=False)

## Spark execution evidence: DAG vs traditional Map -> Shuffle -> Reduce

Classic Hadoop MapReduce executes each job as a rigid two-stage pipeline: a **Map** phase, a **Shuffle** (sort/copy across the network), and a **Reduce** phase. Every job is compiled to exactly this shape, and chaining several MapReduce jobs means writing intermediate results to disk between each one.

Spark instead compiles a chain of transformations into a **Directed Acyclic Graph (DAG)** of stages. Narrow transformations (e.g. `filter`, `withColumn`) are pipelined into a single stage; wide transformations that require a shuffle (e.g. `groupBy`, `orderBy`) introduce a new stage boundary. The DAG scheduler can therefore fuse many logical steps into few physical stages, keep data in memory between them, and use **Adaptive Query Execution (AQE)** to re-optimize the plan at runtime using actual statistics (row counts, partition sizes) rather than only static estimates.

The cells below provide concrete evidence of this execution model for the Spark SQL query defined above:

- `explain(mode="formatted")` — the physical plan, showing the stage-relevant operators (`Exchange` = shuffle boundary, `HashAggregate`, `Scan parquet`, etc.)
- `explain(mode="cost")` — the optimized logical plan with size/row estimates used for planning
- `SparkContext.statusTracker()` — the actual job and stage IDs created when the query runs, which map directly to the DAG stages visible in the Spark UI
- the Spark UI URL, where the same DAG can be inspected visually under the **Jobs** and **SQL** tabs


In [ ]:
# Physical plan: stage boundaries are marked by "Exchange" (shuffle) nodes
spark_sql_result.explain(mode="formatted")

# Optimized logical plan with size/row estimates used by the query planner
spark_sql_result.explain(mode="cost")

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


In [ ]:
# Trigger execution (an action) and capture the job/stage IDs Spark created
# for the DAG, as concrete evidence beyond the printed plan.
spark_sql_result.count()

tracker = spark.sparkContext.statusTracker()
job_ids = tracker.getJobIdsForGroup()
print(f"Job IDs: {job_ids}")

for job_id in job_ids:
    job_info = tracker.getJobInfo(job_id)
    if job_info is None:
        continue
    print(f"Job {job_id} -> stage IDs: {job_info.stageIds}")
    for stage_id in job_info.stageIds:
        stage_info = tracker.getStageInfo(stage_id)
        if stage_info is not None:
            print(
                f"  Stage {stage_id}: {stage_info.numTasks} tasks, "
                f"name='{stage_info.name}'"
            )


### Interpretation

- Spark uses **lazy evaluation**: none of `groupBy`, `agg`, `filter` or the SQL query above ran anything until the `.count()`/`.show()` action was called.
- The `Exchange` operators visible in the formatted physical plan mark the actual **shuffle boundaries** — i.e. the stage splits in the DAG. Everything between two `Exchange` nodes is pipelined into one stage and executed as a set of parallel tasks, not as a separate Map/Reduce job.
- The job/stage IDs above map directly onto what is visible in the **Spark UI** (Jobs, Stages and SQL tabs at the `Spark UI` URL printed earlier) — this is the DAG execution graph, generated and optimized once per query rather than fixed per job.
- This is fundamentally different from Hadoop MapReduce, where each job is hard-wired to one Map phase, one Shuffle, and one Reduce phase, with intermediate results always written to HDFS between chained jobs. Spark's DAG scheduler decides the number and shape of stages dynamically based on the query, and Adaptive Query Execution can still change the plan (e.g. coalescing shuffle partitions, converting a sort-merge join to a broadcast join) after execution has started, using runtime statistics that a static MapReduce plan does not have access to.

